# Stage2 Hazel · 统一补充评价（21个固定模型）
按《Hazel_下一项任务_统一评价_20261026.md》与随附锁定协议执行。

只做学生分类头推理，不训练、不重新推理CLIP/Teacher、不拟合温度、不做bootstrap。
先完成输入核验与21个开发集重放，再评价历史1000条与“补充200条”。

建议选择 **T4 GPU**，PyTorch **2.11.0+cu130**。本笔记本不自动升级/降级环境。
如需CPU，在下方明确设置 DEVICE='cpu'，要求PyTorch2.11.0并通过全部重放。
第一次按顺序运行全部单元格；断线后重新连接，再从头运行。运行设备或代码改变时使用新WORK。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
DRIVE = Path('/content/drive/MyDrive')
WORK = DRIVE / 'Stage2_Hazel_UnifiedEvaluation_20261026'
DEVICE = 'cuda'  # CPU备选：'cpu'；同一WORK不能混用设备
# 通常无需修改。自动搜索整个MyDrive下的子文件夹，不要求ZIP放在根目录。
# 若文件被改名或希望指定位置，填入对应的实际完整路径：
OVERRIDES = {
    'Stage2_统一评价_协议与输入_20261026.zip': '',
    'Stage2_Hazel_results_f54a0fef.zip': '',
    'Stage2_Hazel_TextOnly.zip': '',
}
import numpy as np, pandas as pd, torch
print('torch:', torch.__version__, 'numpy:', np.__version__)
print('设备:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' and torch.cuda.is_available() else DEVICE)


## 1. 写入内嵌评价代码
无需另外上传Python脚本。此步骤只写代码，不运行模型。

In [ ]:
SOURCE_CODE = '"""Frozen Stage2 inference only. No optimizer, fitting, or checkpoint selection."""\nfrom pathlib import Path, PurePosixPath\nfrom contextlib import contextmanager\nimport os, sys, io, json, hashlib, zipfile, subprocess, platform, traceback, datetime, uuid\nimport numpy as np\nimport pandas as pd\n\nPROTOCOL_SHA = \'f4e5acb3ca81cf8a2c27bd347b337c29cfaec9e6717c0793574ccd0a659ffe4e\'\nVERIFIER_SHA = \'a40414480755899ccdf7a4a11622f2c2b531e46f25608a642fd3529f5c205962\'\nNAMES = [\'Stage2_统一评价_协议与输入_20261026.zip\', \'Stage2_Hazel_results_f54a0fef.zip\', \'Stage2_Hazel_TextOnly.zip\']\nSEEDS = [20261010, 20261011, 20261012]\nMETHODS = [\'CE\', \'LS005\', \'LS010\', \'KD_raw\', \'KD_calibrated\', \'Text_CE\', \'Text_KD_raw\']\nMETRICS = [\'macro_f1\', \'accuracy\', \'f1_E\', \'f1_N\', \'f1_C\', \'nll\', \'ece10\', \'brier\']\nBOUNDARY = (\'本轮为已查看过成绩的数据上的后续补充评价，不能解释为首次独立确认。\'\n            \'补充200张中130张曾进入早期验证准备，历史1000张中601张曾进入早期测试准备；\'\n            \'准备清单不证明每条实际运行过，包外完整使用史仍未完全确认。\'\n            \'三种子概率未做集成，重复概率行不构成独立样本量。人工标注继续暂停。\')\n\ndef check(ok, message):\n    if not ok: raise ValueError(message)\n\ndef sha_bytes(b): return hashlib.sha256(b).hexdigest()\ndef sha(p):\n    h = hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024*1024), b\'\'): h.update(b)\n    return h.hexdigest()\ndef read(p): return json.loads(Path(p).read_text(encoding=\'utf-8-sig\'))\ndef atomic(p, b):\n    p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)\n    tmp = p.with_name(p.name + \'.\' + uuid.uuid4().hex + \'.partial\')\n    tmp.write_bytes(b); tmp.replace(p)\ndef save(p, obj): atomic(p, json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False).encode(\'utf-8\'))\ndef csv_save(p, df): atomic(p, df.to_csv(index=False, float_format=\'%.17g\').encode(\'utf-8-sig\'))\ndef log(work, event, **data):\n    p = Path(work)/\'events.jsonl\'; p.parent.mkdir(parents=True, exist_ok=True)\n    with p.open(\'a\', encoding=\'utf-8\') as f:\n        f.write(json.dumps(dict(time=datetime.datetime.now(datetime.timezone.utc).isoformat(), event=event, **data), ensure_ascii=False)+\'\\n\')\n\n@contextmanager\ndef guarded(work, stage):\n    work = Path(work); work.mkdir(parents=True, exist_ok=True)\n    lock = work/\'ACTIVE.lock\'\n    with lock.open(\'x\', encoding=\'utf-8\') as f: f.write(\'进程运行中；仅确认旧进程已结束后可移除此锁。\')\n    log(work, \'start\', stage=stage)\n    try:\n        yield\n        log(work, \'success\', stage=stage)\n    except BaseException:\n        log(work, \'failure\', stage=stage, traceback=traceback.format_exc())\n        raise\n    finally: lock.unlink()\n\ndef discover(root, overrides=None):\n    """One directory walk, including nested Drive folders; never guess a match."""\n    root = Path(root); overrides = overrides or {}\n    check(root.is_dir(), f\'云盘未挂载：{root}\')\n    found = {n: [] for n in NAMES}\n    for n, value in overrides.items():\n        check(n in found, f\'未知输入：{n}\')\n        if value:\n            p = Path(value); check(p.is_file(), f\'手动指定的文件不存在：{p}\'); found[n] = [p]\n    errors = []\n    for folder, dirs, files in os.walk(root, onerror=lambda e: errors.append(str(e))):\n        dirs[:] = [d for d in dirs if d not in [\'.Trash\', \'.ipynb_checkpoints\']]\n        for n in set(files).intersection(NAMES):\n            if not overrides.get(n): found[n].append(Path(folder)/n)\n    for n, paths in found.items():\n        print(n, \'\\n\' + \'\\n\'.join(\'  \'+str(p) for p in paths), flush=True)\n        check(paths, f\'未找到 {n}。请确认挂载账号或在 OVERRIDES 中填写实际完整路径。扫描错误：{errors[:3]}\')\n    return found\n\ndef zip_member(z, suffix):\n    names = [n for n in z.namelist() if n == suffix or n.endswith(\'/\'+suffix)]\n    check(len(names)==1, f\'压缩包条目不唯一或缺失：{suffix}\'); return names[0]\n\ndef prepare(found, work):\n    work = Path(work).resolve()\n    with guarded(work, \'verify_inputs\'):\n        protocol_paths = []\n        for p in found[NAMES[0]]:\n            with zipfile.ZipFile(p) as z:\n                if sha_bytes(z.read(zip_member(z, \'evaluation_protocol_LOCKED.json\'))) == PROTOCOL_SHA:\n                    protocol_paths.append(p)\n        check(protocol_paths, \'没有匹配当前锁定协议的输入包，禁止替换协议继续。\')\n        check(len({sha(p) for p in protocol_paths})==1, \'协议包存在不同内容的副本，请用 OVERRIDES 指定。\')\n        bundle = work/\'inputs\'; bundle.mkdir(exist_ok=True)\n        with zipfile.ZipFile(protocol_paths[0]) as z:\n            anchor = zip_member(z, \'evaluation_protocol_LOCKED.json\')\n            prefix = anchor[:-len(\'evaluation_protocol_LOCKED.json\')]\n            for n in z.namelist():\n                if n.endswith(\'/\'): continue\n                check(n.startswith(prefix), \'协议包目录结构异常\')\n                rel = n[len(prefix):]; parts = PurePosixPath(rel)\n                check(not parts.is_absolute() and \'..\' not in parts.parts and \'\\\\\' not in rel and \':\' not in rel, \'不安全的ZIP路径\')\n                dest = bundle/rel; check(dest.resolve().is_relative_to(bundle.resolve()), \'ZIP路径越界\')\n                b = z.read(n)\n                if dest.exists(): check(dest.read_bytes()==b, f\'已保存输入发生变化：{rel}\')\n                else: atomic(dest,b)\n        check(sha(bundle/\'verify_bundle.py\')==VERIFIER_SHA, \'verify_bundle.py 与已审阅版本不符\')\n        result = subprocess.run([sys.executable, str(bundle/\'verify_bundle.py\')], capture_output=True, text=True, encoding=\'utf-8\')\n        atomic(work/\'verify_bundle.log\', (result.stdout+\'\\n\'+result.stderr).encode(\'utf-8\'))\n        check(result.returncode==0, \'协议包核验失败，详见 verify_bundle.log\')\n        print(result.stdout)\n        protocol = read(bundle/\'evaluation_protocol_LOCKED.json\'); models = read(bundle/\'models_LOCKED.json\')\n        check(len(models)==21 and {m[\'model_id\'] for m in models}=={f\'{a}_{s}\' for a in METHODS for s in SEEDS}, \'模型组或种子错误\')\n        for m in models:\n            text = m[\'method\'].startswith(\'Text_\')\n            check(m[\'label_order\']==\'E/N/C\' and (m[\'head_input\'],m[\'head_hidden\'],m[\'head_output\'],m[\'input_slice\']) == ((512,2048,3,\'512:1024\') if text else (2048,512,3,\'0:2048\')), \'结构/标签顺序不符\')\n            t = m[\'student_temperature\']\n            check(t is None if text else (isinstance(t,(int,float)) and np.isfinite(t) and t>0), \'固定学生温度错误\')\n        audit = []\n        for archive in NAMES[1:]:\n            subset = [m for m in models if m[\'checkpoint_archive\']==archive]\n            valid = []\n            for path in found[archive]:\n                try:\n                    with zipfile.ZipFile(path) as z:\n                        check(len(z.namelist())==len(set(z.namelist())), \'ZIP有重复条目\')\n                        for m in subset:\n                            check(sha_bytes(z.read(m[\'checkpoint_entry\']))==m[\'checkpoint_sha256\'], \'checkpoint SHA256不匹配：\'+m[\'model_id\'])\n                    valid.append(path)\n                except Exception:\n                    log(work, \'rejected_archive\', path=str(path), traceback=traceback.format_exc())\n            check(valid, f\'没有符合全部模型哈希的 {archive}，详见 events.jsonl\')\n            # All accepted copies contain the exact same locked checkpoints.\n            with zipfile.ZipFile(valid[0]) as z:\n                for m in subset:\n                    dest = work/\'checkpoints\'/f\'{m["model_id"]}.pt\'\n                    atomic(dest, z.read(m[\'checkpoint_entry\']))\n                    audit.append(dict(model_id=m[\'model_id\'], archive=str(valid[0]), entry=m[\'checkpoint_entry\'], sha256=sha(dest), expected=m[\'checkpoint_sha256\'], status=\'PASS\'))\n        save(work/\'checkpoint_audit.json\',audit)\n        save(work/\'input_audit.json\',dict(protocol_sha256=PROTOCOL_SHA, protocol_archive=str(protocol_paths[0]),\n              protocol_archive_sha256=sha(protocol_paths[0]), files={str(p.relative_to(bundle)):sha(p) for p in bundle.rglob(\'*\') if p.is_file()},\n              checkpoint_count=len(audit), status=\'PASS\'))\n    return dict(work=work, bundle=bundle, models=models, protocol=protocol)\n\ndef environment(device):\n    import torch\n    check(device in [\'cuda\',\'cpu\'], \'DEVICE 只能为 cuda 或 cpu\')\n    check(device!=\'cuda\' or torch.cuda.is_available(), \'未检测到GPU：选择Colab T4运行时，或明确设置 DEVICE="cpu"。\')\n    check(str(torch.__version__).split(\'+\')[0]==\'2.11.0\', \'协议要求 PyTorch 2.11.0；请先恢复约定版本。\')\n    if device==\'cuda\': check(torch.version.cuda==\'13.0\', \'协议GPU环境为cu130，请先核查环境。\')\n    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False\n    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n    torch.use_deterministic_algorithms(True)\n    return dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, torch=str(torch.__version__),\n                device=device, gpu=torch.cuda.get_device_name(0) if device==\'cuda\' else None, cuda=torch.version.cuda,\n                cudnn=torch.backends.cudnn.version(), dtype=\'float32\', tf32=False, deterministic=True, batch_size=256)\n\ndef load_data(bundle, split):\n    name = \'val_select_replay\' if split==\'val_select\' else split\n    with np.load(bundle/f\'features/{name}.npz\', allow_pickle=False) as a: d = {k:a[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n    rows = [json.loads(s) for s in (bundle/f\'manifests/{split}.jsonl\').read_text(encoding=\'utf-8-sig\').splitlines() if s.strip()]\n    n = {\'val_select\':400,\'historical_test\':1000,\'new_dev_holdout\':200}[split]\n    check(d[\'x\'].shape==(n,2048) and d[\'x\'].dtype==np.float32 and np.isfinite(d[\'x\']).all(), \'特征维度/精度/有限性不符\')\n    check(d[\'ids\'].tolist()==[r[\'sample_id\'] for r in rows] and d[\'y\'].tolist()==[r[\'label\'] for r in rows], \'特征与清单ID/标签错位\')\n    check(d[\'image_ids\'].astype(str).tolist()==[str(r[\'image_id\']) for r in rows] and len(set(d[\'ids\']))==n, \'图片ID/样本唯一性错误\')\n    return d\n\ndef infer(ctx, m, d, device):\n    import torch\n    from torch import nn\n    path = ctx[\'work\']/\'checkpoints\'/f\'{m["model_id"]}.pt\'\n    check(sha(path)==m[\'checkpoint_sha256\'], \'推理前checkpoint哈希不符\')\n    model = nn.Sequential(nn.Linear(m[\'head_input\'],m[\'head_hidden\']),nn.ReLU(),nn.Dropout(m[\'dropout\']),nn.Linear(m[\'head_hidden\'],3))\n    model.load_state_dict(torch.load(path,map_location=\'cpu\',weights_only=True),strict=True)\n    model = model.to(device=device,dtype=torch.float32).eval()\n    start,stop = map(int,m[\'input_slice\'].split(\':\'))\n    x = np.ascontiguousarray(d[\'x\'][:,start:stop],dtype=np.float32)\n    with torch.inference_mode():\n        z = np.concatenate([model(torch.from_numpy(x[i:i+256]).to(device)).cpu().numpy() for i in range(0,len(x),256)])\n    check(z.shape==(len(x),3) and np.isfinite(z).all(), \'推理输出异常\')\n    return z\n\ndef replay(ctx, device):\n    work = ctx[\'work\']\n    with guarded(work, \'replay_all_21\'):\n        env = environment(device)\n        config = dict(environment=env, protocol_sha256=PROTOCOL_SHA, model_registry_sha256=sha(ctx[\'bundle\']/\'models_LOCKED.json\'),\n                      code_sha256=sha(__file__), inference_only=True, replay_tolerance=1e-4, label_order=[\'E\',\'N\',\'C\'])\n        locked = work/\'run_config.json\'\n        if locked.exists(): check(read(locked)==config, \'本轮环境或代码发生变化，请新建独立 WORK，不能混用缓存。\')\n        else: save(locked,config)\n        d = load_data(ctx[\'bundle\'],\'val_select\'); records=[]\n        for m in ctx[\'models\']:\n            z = infer(ctx,m,d,device)\n            r = pd.read_csv(ctx[\'bundle\']/f\'replay_reference/{m["model_id"]}.csv\',dtype={\'sample_id\':str})\n            check(not r.sample_id.duplicated().any() and set(r.sample_id)==set(d[\'ids\']), \'重放参考样本ID不一致\')\n            r = r.set_index(\'sample_id\').loc[d[\'ids\']]\n            check(np.array_equal(r.gold.to_numpy(),d[\'y\']), \'重放参考标签不一致\')\n            diff = float(np.max(np.abs(z.astype(np.float64)-r[[\'logit_E\',\'logit_N\',\'logit_C\']].to_numpy())))\n            mismatches = int(np.sum(z.argmax(1)!=r.prediction.to_numpy()))\n            ok = mismatches==0 and diff<=1e-4\n            records.append(dict(model_id=m[\'model_id\'], checkpoint_sha256=m[\'checkpoint_sha256\'], max_abs_logit_diff=diff,\n                                prediction_mismatches=mismatches, rows=400, status=\'PASS\' if ok else \'FAIL\'))\n            csv_save(work/\'replay_audit.csv\',pd.DataFrame(records))\n            out = pd.DataFrame(dict(sample_id=d[\'ids\'],gold=d[\'y\'],prediction=z.argmax(1)))\n            for i,c in enumerate(\'ENC\'): out[\'logit_\'+c]=z[:,i]\n            csv_save(work/\'replay_predictions\'/f\'{m["model_id"]}.csv\',out)\n            print(\'重放\',m[\'model_id\'],records[-1][\'status\'],f\'最大差={diff:.3g}\',flush=True)\n            check(ok, \'开发集重放失败；评价尚未启动。详见 replay_audit.csv，不能放宽容差。\')\n        save(work/\'replay_PASS.json\',dict(count=21,run_config_sha256=sha(locked),audit_sha256=sha(work/\'replay_audit.csv\')))\n    ctx[\'device\']=device; ctx[\'config\']=config\n    return pd.DataFrame(records)\n\ndef scores(z, y, temperature=1.):\n    check(np.isfinite(temperature) and temperature>0, \'温度必须为有限正数\')\n    a = np.asarray(z,dtype=np.float64)/temperature\n    a -= a.max(axis=1,keepdims=True)\n    logp = a-np.log(np.exp(a).sum(axis=1,keepdims=True)); p = np.exp(logp)\n    pred = np.argmax(a,axis=1); y=np.asarray(y,dtype=np.int64)\n    f1=[]\n    for c in range(3):\n        tp=np.sum((pred==c)&(y==c)); den=np.sum(pred==c)+np.sum(y==c)\n        f1.append(float(2*tp/den) if den else 0.)\n    conf=p.max(axis=1); correct=(pred==y); bins=np.minimum(np.floor(conf*10).astype(int),9)\n    ece=0.\n    for k in range(10):\n        ix=bins==k\n        if ix.any(): ece+=float(ix.mean()*abs(correct[ix].mean()-conf[ix].mean()))\n    result=dict(macro_f1=float(np.mean(f1)),accuracy=float(correct.mean()),f1_E=f1[0],f1_N=f1[1],f1_C=f1[2],\n                nll=float(-logp[np.arange(len(y)),y].mean()),ece10=ece,brier=float(np.square(p-np.eye(3)[y]).sum(axis=1).mean()))\n    return p,pred,result\n\ndef evaluate(ctx):\n    work=ctx[\'work\']; device=ctx[\'device\']\n    with guarded(work,\'evaluate_frozen_models\'):\n        check(environment(device)==ctx[\'config\'][\'environment\'], \'设备或环境改变\')\n        check(read(work/\'run_config.json\')==ctx[\'config\'], \'运行配置改变\')\n        receipt=read(work/\'replay_PASS.json\')\n        check(receipt[\'count\']==21 and receipt[\'run_config_sha256\']==sha(work/\'run_config.json\') and receipt[\'audit_sha256\']==sha(work/\'replay_audit.csv\'), \'未通过全部21模型重放\')\n        expected=read(work/\'input_audit.json\')[\'files\']\n        for n,h in expected.items(): check(sha(ctx[\'bundle\']/n)==h, \'评价前输入发生变化：\'+n)\n        metrics=[]; raw_tables=[]; probability_tables=[]\n        for split in [\'historical_test\',\'new_dev_holdout\']:\n            d=load_data(ctx[\'bundle\'],split)\n            for m in ctx[\'models\']:\n                name=m[\'model_id\']; path=work/\'raw_cache\'/split/f\'{name}.npz\'; rp=path.with_suffix(\'.json\')\n                key=dict(config_sha256=sha(work/\'run_config.json\'),checkpoint_sha256=m[\'checkpoint_sha256\'],\n                         feature_sha256=sha(ctx[\'bundle\']/f\'features/{split}.npz\'),split=split,model_id=name)\n                if path.exists() and rp.exists():\n                    r=read(rp); check(r[\'key\']==key and r[\'sha256\']==sha(path), \'原始预测缓存发生变化，禁止静默复用\')\n                    with np.load(path,allow_pickle=False) as a: z=a[\'logits\'].copy()\n                    log(work,\'reuse_verified_logits\',split=split,model_id=name)\n                else:\n                    if path.exists() or rp.exists(): log(work,\'retry_incomplete_cache\',split=split,model_id=name)\n                    z=infer(ctx,m,d,device); b=io.BytesIO(); np.savez_compressed(b,logits=z)\n                    atomic(path,b.getvalue()); save(rp,dict(key=key,sha256=sha(path)))\n                check(z.shape==(len(d[\'y\']),3) and z.dtype==np.float32 and np.isfinite(z).all(),\'缓存输出异常\')\n                base=pd.DataFrame(dict(split=split,model_id=name,method=m[\'method\'],seed=m[\'seed\'],sample_id=d[\'ids\'],\n                                       image_id=d[\'image_ids\'],gold=d[\'y\'],checkpoint_sha256=m[\'checkpoint_sha256\']))\n                for i,c in enumerate(\'ENC\'): base[\'logit_\'+c]=z[:,i]\n                variants=[(\'raw\',1.)]\n                if m[\'student_temperature\'] is not None: variants.append((\'post_temperature\',m[\'student_temperature\']))\n                raw_metric=None\n                for variant,t in variants:\n                    p,pred,metric=scores(z,d[\'y\'],t)\n                    check(np.array_equal(pred,z.argmax(1)), \'温度后处理改变预测类别\')\n                    if raw_metric is None: raw_metric=metric\n                    else:\n                        check(all(metric[k]==raw_metric[k] for k in [\'macro_f1\',\'accuracy\',\'f1_E\',\'f1_N\',\'f1_C\']), \'温度后处理改变分类指标\')\n                    table=base.copy(); table[\'prediction\']=pred; table[\'variant\']=variant; table[\'student_T\']=t\n                    for i,c in enumerate(\'ENC\'): table[\'p_\'+c]=p[:,i]\n                    table=table[ctx[\'protocol\'][\'outputs\'][\'required_prediction_fields\']]\n                    probability_tables.append(table)\n                    if variant==\'raw\': raw_tables.append(table)\n                    metrics.append(dict(split=split,display_name=\'历史1000条\' if split==\'historical_test\' else \'补充200条\',\n                                        model_id=name,method=m[\'method\'],seed=m[\'seed\'],variant=variant,student_T=t,\n                                        checkpoint_sha256=m[\'checkpoint_sha256\'],n=len(d[\'y\']),**metric))\n                print(\'完成\',split,name,flush=True)\n        raw=pd.concat(raw_tables,ignore_index=True); probs=pd.concat(probability_tables,ignore_index=True); per=pd.DataFrame(metrics)\n        check(len(raw)==25200 and len(probs)==43200 and len(per)==72,\'固定输出数量不符\')\n        check(not probs.duplicated([\'split\',\'model_id\',\'variant\',\'sample_id\']).any(),\'逐条输出有重复主键\')\n        summary=[]\n        for keys,g in per.groupby([\'split\',\'display_name\',\'method\',\'variant\'],sort=False):\n            check(len(g)==3 and sorted(g.seed)==SEEDS,\'三种子不完整，不能生成完整汇总\')\n            row=dict(zip([\'split\',\'display_name\',\'method\',\'variant\'],keys)); row[\'seed_count\']=3\n            for k in METRICS: row[k+\'_mean\']=float(g[k].mean()); row[k+\'_sd\']=float(g[k].std(ddof=1))\n            summary.append(row)\n        summary=pd.DataFrame(summary)\n        for split in [\'historical_test\',\'new_dev_holdout\']:\n            out=work/\'report\'/split\n            csv_save(out/\'raw_predictions.csv\',raw[raw.split==split])\n            csv_save(out/\'all_probabilities.csv\',probs[probs.split==split])\n            csv_save(out/\'metrics_per_seed.csv\',per[per.split==split])\n            csv_save(out/\'metrics_mean_sampleSD.csv\',summary[summary.split==split])\n        save(work/\'completion.json\',dict(status=\'COMPLETE\',raw_predictions=25200,probability_rows=43200,\n             raw_model_split_outputs=42,total_variant_split_outputs=72,training_performed=False,temperature_fitted=False,\n             bootstrap_performed=False,protocol_sha256=PROTOCOL_SHA,interpretation=BOUNDARY))\n        atomic(work/\'解释边界.txt\',BOUNDARY.encode(\'utf-8\'))\n    return summary\n\ndef package(ctx):\n    work=ctx[\'work\']; check(read(work/\'completion.json\')[\'status\']==\'COMPLETE\',\'尚未完整完成评价\')\n    with guarded(work,\'package\'):\n        atomic(work/\'code/unified_eval.py\',Path(__file__).read_bytes())\n    # Models stay in WORK for reproducibility; don\'t duplicate the original model ZIPs in delivery.\n    paths=[p for p in work.rglob(\'*\') if p.is_file() and \'checkpoints\' not in p.relative_to(work).parts and p.name not in [\'PACKAGE_SHA256.json\',\'ACTIVE.lock\'] and not p.name.endswith(\'.partial\')]\n    save(work/\'PACKAGE_SHA256.json\',{p.relative_to(work).as_posix():sha(p) for p in sorted(paths)})\n    target=work.parent/(work.name+\'_results.zip\'); tmp=target.with_suffix(\'.zip.partial\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in paths+[work/\'PACKAGE_SHA256.json\']: z.write(p,p.relative_to(work).as_posix())\n    tmp.replace(target)\n    with zipfile.ZipFile(target) as z:\n        for n,h in json.loads(z.read(\'PACKAGE_SHA256.json\')).items(): check(sha_bytes(z.read(n))==h,\'结果包校验失败：\'+n)\n    print(\'结果包：\',target,\'\\nSHA256：\',sha(target)); return target\n'
import importlib.util, hashlib
WORK.mkdir(parents=True, exist_ok=True)
CODE = WORK / 'code' / 'unified_eval.py'
CODE.parent.mkdir(exist_ok=True)
if CODE.exists() and CODE.read_text(encoding='utf-8') != SOURCE_CODE:
    raise RuntimeError('已有WORK中的代码与当前版本不同，请使用新的WORK目录。')
CODE.write_text(SOURCE_CODE, encoding='utf-8', newline='\n')
spec = importlib.util.spec_from_file_location('unified_eval', CODE)
ue = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ue)
print('代码已就绪：', CODE)


## 2. 自动定位输入并核验
搜索可能需要一会儿，会打印文件实际路径。同名副本按锁定checkpoint哈希核验，不按文件名盲选。协议包中的verify_bundle.py也会运行；本步骤无需GPU。

In [ ]:
found = ue.discover(DRIVE, OVERRIDES)
ctx = ue.prepare(found, WORK)
print('输入检查通过：21个checkpoint，15个固定学生温度。')
display(pd.DataFrame(ctx['models'])[['model_id','seed','head_input','head_hidden','student_temperature']])


## 3. 全部21个模型开发集重放
400条val_select，预测类别全部一致，原始logits最大绝对差≤1e-4。若失败立即停下，记录保留在WORK中；不要改容差。

In [ ]:
replay_audit = ue.replay(ctx, DEVICE)
display(replay_audit)

## 4. 两套数据分别评价
每个模型/数据组合只推理一次；后温度结果直接使用原始logits。
共42组原始预测（25200行）、72组概率结果（43200行）。
21个模型全部报告raw，15个图文模型另报告post_temperature，文本模型不拟合温度。
保持eval、FP32、batch256、关闭TF32。此单元格不使用训练τ=2。
断线重跑会复用与当前输入、环境、代码完全匹配的预测缓存；若留下ACTIVE.lock，仅在确认旧任务已结束后手动删除该锁文件。


In [ ]:
summary = ue.evaluate(ctx)
for split, title in [('historical_test', '历史1000条'), ('new_dev_holdout', '补充200条')]:
    print(title)
    display(summary[summary['split'] == split])


## 5. 完整打包
输出两套独立结果表、全部逐条输出、协议与哈希、模型核验、重放记录、运行配置、失败/重试日志和代码。原始模型保留在WORK/checkpoints，结果包用精确哈希引用，避免重复打包模型。

In [ ]:
RESULT_ZIP = ue.package(ctx)
print('已保存在Google Drive：', RESULT_ZIP)
print(ue.BOUNDARY)
# 如需要立即下载到电脑，取消下面两行的注释：
# from google.colab import files
# files.download(str(RESULT_ZIP))


## 交付和解释
将本notebook与unified_eval.py上传Stage2_Hazel分支（由Hazel自行上传），把生成的results.zip发回。
主要比较固定为历史1000条的图文原始KD−图文CE Macro-F1，其余为辅助分析；配对区间与错误分析由芙芙完成。

两套数据均已查看过历史成绩，本轮不能称为首次独立确认。补充200张中130张曾进入早期验证准备，历史1000张中601张曾进入早期测试准备；准备清单不证明逐条实际运行过，包外完整使用史仍未完全确认。
不根据本轮成绩改变温度、α、checkpoint或种子数量。所有指标CSV使用0到1小数，三种子均值与样本标准差不等于概率集成。
